In [6]:
%pip install playwright beautifulsoup4 pandas tqdm
!playwright install firefox


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [7]:
import asyncio
import pandas as pd
import re
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright

BASE_URL = "https://www.congress.gov/search?q=%7B%22source%22%3A%22legislation%22%2C%22type%22%3A%22bills%22%2C%22house-committee%22%3A%22Science%2C+Space%2C+and+Technology%22%7D"

OUTPUT_FILE = "bills.csv"

In [8]:
def parse_sponsor(text):
    if not text:
        return {}

    name_match = re.search(r"Sponsor:\s*(.*?)\s*\[", text)
    party_match = re.search(r"\[Rep\.-(R|D|I)-([A-Z]{2})-(\d+)\]", text)
    cosponsors_match = re.search(r"Cosponsors:\s*\(\s*(\d+)\s*\)", text)
    date_match = re.search(r"\(Introduced\s*([0-9/]+)\)", text)

    return {
        "sponsor_name": name_match.group(1).strip() if name_match else None,
        "party": party_match.group(1) if party_match else None,
        "state": party_match.group(2) if party_match else None,
        "district": party_match.group(3) if party_match else None,
        "cosponsors": int(cosponsors_match.group(1)) if cosponsors_match else 0,
        "introduced_date": date_match.group(1) if date_match else None,
    }


def clean_status(text):
    if not text:
        return None
    return text.split("Array")[0].strip()

In [9]:
def parse_bill(li):
    heading = li.select_one("span.result-heading a")
    bill_number = heading.get_text(strip=True) if heading else None
    bill_url = "https://www.congress.gov" + heading["href"] if heading else None

    title = li.select_one("span.result-title")
    title = title.get_text(" ", strip=True) if title else None

    sponsor_raw = None
    committees = None
    latest_action = None

    for item in li.select("span.result-item"):
        txt = item.get_text(" ", strip=True)
        if "Sponsor:" in txt:
            sponsor_raw = txt
        elif "Committees:" in txt:
            committees = txt
        elif "Latest Action:" in txt:
            latest_action = txt

    status_el = li.select_one("ol.stat_leg li.selected")
    status = clean_status(status_el.get_text(" ", strip=True) if status_el else None)

    sponsor_data = parse_sponsor(sponsor_raw)

    return {
        "bill_number": bill_number,
        "bill_url": bill_url,
        "title": title,
        "committees": committees,
        "latest_action": latest_action,
        "status": status,
        **sponsor_data,
    }

In [10]:
async def scrape_all():
    all_data = []
    page_num = 1

    async with async_playwright() as p:
        browser = await p.firefox.launch(headless=False)
        page = await browser.new_page()

        while True:
            url = BASE_URL + f"&page={page_num}"
            print(f"Scraping page {page_num}...")

            await page.goto(url, wait_until="networkidle", timeout=120000)

            soup = BeautifulSoup(await page.content(), "html.parser")
            bills = soup.select("li.expanded")

            if len(bills) == 0:
                print("No more results. stopping.")
                break

            for b in bills:
                all_data.append(parse_bill(b))

            print(f"  → {len(bills)} bills (total {len(all_data)})")

            page_num += 1

        await browser.close()

    return all_data

In [11]:
data = await scrape_all()

print("Total scraped:", len(data))
print(data[0])

Scraping page 1...
  → 100 bills (total 100)
Scraping page 2...
  → 100 bills (total 200)
Scraping page 3...
No more results. stopping.
Total scraped: 200
{'bill_number': 'H.R.9154', 'bill_url': 'https://www.congress.gov/bill/119th-congress/house-bill/9154?s=1&r=1', 'title': 'To direct the Secretary of Commerce to develop a methodology for identifying country of origin of shrimp, and for other purposes.', 'committees': 'Committees: House - Natural Resources; Science, Space, and Technology', 'latest_action': 'Latest Action: House - 06/04/2026 Referred to the Committee on Natural Resources, and in addition to the Committee on Science, Space, and Technology , for a period to be subsequently determined by the Speaker, in each case for consideration of such provisions as fall within the jurisdiction of the... ( All Actions )', 'status': 'Introduced', 'sponsor_name': 'Mace, Nancy', 'party': 'R', 'state': 'SC', 'district': '1', 'cosponsors': 3, 'introduced_date': '06/04/2026'}


In [12]:
import os

df_new = pd.DataFrame(data)

# Load existing file if it exists
if os.path.exists(OUTPUT_FILE):
    df_old = pd.read_csv(OUTPUT_FILE)

    # append-only (no dedup logic required)
    df_final = pd.concat([df_old, df_new], ignore_index=True)
else:
    df_final = df_new

df_final.to_csv(OUTPUT_FILE, index=False)

print(f"Saved {len(df_final)} total rows to {OUTPUT_FILE}")

Saved 200 total rows to bills.csv
